# Week 46, Day 1: Binary search trees

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w046d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))

def search(node, target):
    path = []
    while node is not None:
        path.append(node.val)
        if target == node.val:
            return True, path
        node = node.left if target < node.val else node.right
    return False, path

print(search(root, 35))
print(search(root, 65))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def contains(node, target):
    while node is not None:
        if target == node.val:
            return True
        node = node.left if target < node.val else node.right
    return False

def minimum(node):
    while node.left is not None:
        node = node.left
    return node.val

def maximum(node):
    while node.right is not None:
        node = node.right
    return node.val

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
print(contains(root, 60), contains(root, 61), minimum(root), maximum(root), minimum(root.right))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_bst(root):
    stack = [(root, float("-inf"), float("inf"))]
    while stack:
        node, low, high = stack.pop()
        if node is None:
            continue
        if not low < node.val < high:
            return False
        stack.append((node.left, low, node.val))
        stack.append((node.right, node.val, high))
    return True

T = TreeNode
print(is_bst(T(10, T(5), T(15, T(12), T(20)))), is_bst(T(10, T(5), T(15, T(6), T(20)))), is_bst(None))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def floor(node, x):
    best = None
    while node is not None:
        if node.val == x:
            return x
        if node.val < x:
            best = node.val          # a candidate; something closer may be to the right
            node = node.right
        else:
            node = node.left
    return best

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
print([floor(root, x) for x in [10, 20, 37, 65, 100]])

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def in_range(node, lo, hi, out, visited):
    if node is None:
        return out
    visited.append(node.val)
    if lo < node.val:
        in_range(node.left, lo, hi, out, visited)
    if lo <= node.val <= hi:
        out.append(node.val)
    if node.val < hi:
        in_range(node.right, lo, hi, out, visited)
    return out

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
visited = []
print(in_range(root, 33, 62, [], visited), "visited", len(visited), "of 9")

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def insert(root, val):
    if root is None:
        return TreeNode(val)
    node = root
    while True:
        if val < node.val:
            if node.left is None:
                node.left = TreeNode(val)
                return root
            node = node.left
        else:
            if node.right is None:
                node.right = TreeNode(val)
                return root
            node = node.right

def height(node):
    stack, best = [(node, 0)] if node else [], -1
    while stack:
        n, d = stack.pop()
        best = max(best, d)
        stack.extend((c, d + 1) for c in (n.left, n.right) if c)
    return best

import random
values = list(range(1, 1024))
shuffled = values[:]
random.Random(1).shuffle(shuffled)
for name, order in [("sorted", values), ("shuffled", shuffled)]:
    root = None
    for v in order:
        root = insert(root, v)
    print(f"{name:<9} height {height(root)}")

> **Key idea.** A BST keeps every value in a node's left subtree smaller and every value in its right subtree larger, so in-order traversal is sorted, the minimum is leftmost and the maximum rightmost. Searching goes left or right at each node: O(h). Check a BST by passing lower and upper bounds down (checking only children is wrong). Floor and ceiling remember the best candidate on the search path; range queries skip subtrees that can't hold values in range, for O(h + k). The shape depends on insertion order: sorted input makes a chain (O(n) searches), random input a height of about 3 log₂ n, and building from the middle of sorted values the minimum height.

### Exercise: Searching

Write `search_path(root, target)`: the list of values visited while searching for `target` (ending with `target` if it's found), and `contains(root, target)`. Use loops, not recursion; the checks search a chain 20,000 nodes tall.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def search_path(root, target):
    ___

def contains(root, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_search():\n    \"searching\"\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    assert search_path(root, 35) == [50, 30, 40, 35] and search_path(root, 65) == [50, 70, 60] and search_path(None, 1) == []\n    assert contains(root, 90) and not contains(root, 55) and not contains(None, 1)\n    chain = None\n    for v in range(20_000):\n        chain = T(v, chain)\n    assert contains(chain, 0) and len(search_path(chain, -1)) == 20_000")

### Exercise: Smallest, largest and second

Write `minimum(root)` and `maximum(root)` (raising `ValueError` for an empty tree), and `second_largest(root)` (the second largest value; `ValueError` if there are fewer than two nodes), each in O(h).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def minimum(root):
    ___

def maximum(root):
    ___

def second_largest(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_extremes():\n    \"extremes\"\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    assert minimum(root) == 20 and maximum(root) == 90 and second_largest(root) == 80\n    assert second_largest(T(5, T(3, None, T(4)))) == 4 and second_largest(T(5, T(3), T(8, T(6, None, T(7))))) == 7\n    for f, t in [(minimum, None), (maximum, None), (second_largest, None), (second_largest, T(1))]:\n        try:\n            f(t)\n            assert False\n        except ValueError:\n            pass")

### Exercise: Is it a BST?

Write `is_bst(root)` with bounds passed down (no duplicates allowed), using an explicit stack. Then `first_violation(root)`: the value of the first node, in pre-order, that breaks the property, or `None` if it's a BST.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_bst(root):
    ___

def first_violation(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_valid():\n    \"checking\"\n    assert is_bst(T(10, T(5), T(15, T(12), T(20)))) and is_bst(None) and is_bst(T(1))\n    assert not is_bst(T(10, T(5), T(15, T(6), T(20)))), \"6 is right of 10\"\n    assert not is_bst(T(10, T(5, None, T(11)), T(15))) and not is_bst(T(2, T(2))), \"no duplicates\"\n    assert first_violation(T(10, T(5), T(15, T(6), T(20)))) == 6 and first_violation(T(10, T(12, T(1)), T(5))) == 12 and first_violation(T(1)) is None\n    chain = None\n    for v in range(10_000):\n        chain = T(v, chain)\n    assert is_bst(chain)")

### Exercise: Floor, ceiling and closest

Write `floor(root, x)` (the largest value ≤ `x`), `ceiling(root, x)` (the smallest value ≥ `x`), both `None` if there's none, and `closest(root, x)` (the value nearest to `x`, the smaller one on a tie; `None` for an empty tree). Each walks one path: O(h).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def floor(root, x):
    ___

def ceiling(root, x):
    ___

def closest(root, x):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_nearest():\n    \"nearest values\"\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    assert [floor(root, x) for x in [10, 20, 37, 65, 100]] == [None, 20, 35, 60, 90]\n    assert [ceiling(root, x) for x in [10, 20, 37, 65, 100]] == [20, 20, 40, 70, None]\n    assert [closest(root, x) for x in [0, 37, 45, 65, 1000]] == [20, 35, 40, 60, 90] and closest(None, 5) is None")

### Exercise: Values in a range

Write `in_range(root, lo, hi)`: the values from `lo` to `hi` inclusive, sorted, by an in-order walk with an **explicit stack** that never goes into a subtree that can't contain values in the range. The checks count how many nodes you visit: for a range near one end of a 4,095-node balanced tree, it must be a few dozen, not thousands.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def in_range(root, lo, hi):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Counted:\n    visits = 0\n    def __init__(self, val, left=None, right=None):\n        self._val, self.left, self.right = val, left, right\n    @property\n    def val(self):\n        Counted.visits += 1\n        return self._val\ndef balanced(lo, hi):\n    if lo > hi:\n        return None\n    mid = (lo + hi) // 2\n    return Counted(mid, balanced(lo, mid - 1), balanced(mid + 1, hi))\ndef test_range():\n    \"ranges\"\n    root = balanced(1, 4095)\n    assert in_range(root, 10, 20) == list(range(10, 21)) and in_range(root, 5000, 6000) == [] and in_range(root, 4090, 9999) == list(range(4090, 4096))\n    Counted.visits = 0\n    in_range(root, 10, 20)\n    assert Counted.visits < 150, f\"{Counted.visits} reads of .val\"\n    assert in_range(None, 1, 2) == []")

### Exercise: A balanced BST from sorted values

Write `from_sorted(values)`: build a BST of the smallest possible height from a sorted list of distinct values, choosing the middle value (`(lo + hi) // 2`) as each subtree's root. Work on index ranges, not slices. Then `height(root)` (edges; −1 for empty).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def from_sorted(values):
    ___

def height(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef test_build():\n    \"balanced\"\n    for n in [0, 1, 2, 3, 7, 8, 100, 1000]:\n        root = from_sorted(list(range(n)))\n        assert inorder(root) == list(range(n)) and height(root) == (math.floor(math.log2(n)) if n else -1), n\n    r = from_sorted([1, 2, 3, 4, 5])\n    assert r.val == 3 and r.left.val == 1 and r.right.val == 4")

### Bug hunt: Checking only the children

This checks each node against its own children only, so it misses a value deeper down on the wrong side of an ancestor: this tree has 6 to the right of 10, yet it prints `True`. Pass bounds down instead, so it prints `False`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_bst(node):
    if node is None:
        return True
    if node.left is not None and node.left.val >= node.val:
        return False
    if node.right is not None and node.right.val <= node.val:
        return False
    return is_bst(node.left) and is_bst(node.right)

T = TreeNode
print(is_bst(T(10, T(5), T(15, T(6), T(20)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bounds\"\n    expect_output(\"False\")\n    assert is_bst(TreeNode(10, TreeNode(5), TreeNode(15, TreeNode(12), TreeNode(20))))")

### Bug hunt: Turning the wrong way

This search goes **right** when the target is smaller, so it walks away from the values it should find: this prints `False False`. Fix the direction so it prints `True False`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def contains(node, target):
    while node is not None:
        if target == node.val:
            return True
        node = node.right if target < node.val else node.left
    return False

T = TreeNode
root = T(50, T(30, T(20), T(40)), T(70, T(60), T(80)))
print(contains(root, 20), contains(root, 55))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"smaller goes left\"\n    expect_output(\"True False\")")

### Bug hunt: The last, not the best

This `floor` returns the last node it **visited**, not the last node where it went right, so it can return a value that's too big: the floor of 33 comes out as 35. Remember the best candidate (a value below `x`) as you go, so it prints `30`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def floor(node, x):
    last = None
    while node is not None:
        last = node.val
        if node.val == x:
            return x
        node = node.right if node.val < x else node.left
    return last

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70))
print(floor(root, 33))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the best candidate\"\n    expect_output(\"30\")\n    assert floor(root, 10) is None and floor(root, 45) == 40")

### Puzzle: Searching a BST

Arrange the lines into an iterative BST search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For the BST built by inserting 8, 3, 10, 1, 6, 14, 4, 7, 13 in that order:

p = ___   # its in-order traversal
q = ___   # the values visited searching for 7
r = ___   # its height
s = ___   # the floor of 5
t = ___   # the height after inserting 1, 2, 3, ..., 9 in order into an empty BST

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 3, 4, 6, 7, 8, 10, 13, 14] and q == [8, 3, 6, 7] and r == 3, \"p, q and r\"\n    assert s == 4 and t == 8, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: kth smallest and a pair with a sum

Two classic BST problems that use iterators (Week 45 Day 3) so they stop as soon as they have the answer:

1. `kth_smallest(root, k)`: the kth smallest value (k from 1), by an in-order iterator that stops after k values, so it costs O(h + k). Raise `IndexError` if the tree has fewer than k values.
2. `pair_with_sum(root, target)`: two **different** nodes whose values add up to `target`, as `(smaller, larger)`, or `None`. Use Week 41's two pointers on the tree itself: an ascending in-order iterator and a **descending** one (right, node, left), moving whichever side the sum says, and stopping when they meet. O(n) time and O(h) space: don't build a list of all the values.

Write both iterator classes (`Ascending` and `Descending`) without `yield`. The checks use a balanced tree of 100,000 values and count how many nodes `kth_smallest` touches.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class Ascending:
    ___

class Descending:
    ___

def kth_smallest(root, k):
    ___

def pair_with_sum(root, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef balanced(values, lo, hi):\n    if lo > hi:\n        return None\n    mid = (lo + hi) // 2\n    return T(values[mid], balanced(values, lo, mid - 1), balanced(values, mid + 1, hi))\ndef test_kth():\n    \"kth smallest\"\n    vals = list(range(0, 200_000, 2))\n    root = balanced(vals, 0, len(vals) - 1)\n    assert kth_smallest(root, 1) == 0 and kth_smallest(root, 5) == 8 and kth_smallest(root, 100_000) == 199_998\n    try:\n        kth_smallest(root, 100_001)\n        assert False\n    except IndexError:\n        pass\n    it = Ascending(root)\n    assert len(it._stack) <= 20, \"only the path\"\n    assert [next(Descending(root)) for _ in range(1)] == [199_998]\ndef test_pair():\n    \"pairs\"\n    rng = random.Random(3)\n    for _ in range(100):\n        vals = sorted(rng.sample(range(-50, 50), rng.randint(0, 15)))\n        root = balanced(vals, 0, len(vals) - 1)\n        target = rng.randint(-60, 60)\n        got = pair_with_sum(root, target)\n        pairs = [(a, b) for i, a in enumerate(vals) for b in vals[i + 1:] if a + b == target]\n        assert (got is None) == (not pairs) and (got is None or got in pairs), (vals, target, got)\n    vals = list(range(100_000))\n    big = balanced(vals, 0, len(vals) - 1)\n    assert pair_with_sum(big, 199_997) == (99_998, 99_999)")

## Warm-up

Finding 35 looked at 4 of the 9 nodes; deciding 65 isn't there looked at 3. At each node, the BST rule says which side the target must be on, so the other side is never touched. That's binary search, built into the shape of a tree.

## The BST property

A binary tree is a **binary search tree** when, for **every** node:

- every value in its **left** subtree is **less** than the node's value, and
- every value in its **right** subtree is **greater**.

```text
             50
           /    \
         30      70            every value left of 50 is less than 50,
        /  \    /  \           every value right of 50 is greater, and the
      20   40  60   80         same holds at 30, at 70, at 40, at 80...
           /          \
          35           90
```

Two consequences follow:

- An **in-order** traversal (Week 45 Day 3: left, node, right) visits the values in **sorted** order.
- The smallest value is the **leftmost** node, and the largest the **rightmost**.

What about **duplicates**? This course's BSTs hold each value at most once (like a set); to store repeats, keep a count in the node, or put equal values consistently on one side.

## Searching

Searching starts at the root and, at each node, compares and goes left or right: O(h) for a tree of height h. A loop does it with no recursion (and so no recursion limit):

The cost depends entirely on the height. For a balanced tree of a million values, h is about 20; for a degenerate one, a million. Keeping trees balanced is Day 3's job.

## Checking a BST

It's tempting to check only that each node's children are on the correct sides. That's not enough:

```text
        10
       /  \
      5    15           every child is on the correct side of its parent,
          /  \          but 6 is in 10's right subtree and smaller than 10:
         6    20        this is NOT a BST
```

The correct check passes **bounds** down the tree (Week 45 Day 5's "information flowing down"): each node must lie strictly between a lower and an upper bound. Going left, the node's value becomes the new upper bound; going right, the new lower bound.

Another correct check: walk the tree in order and confirm each value is bigger than the one before.

## Nearest values and ranges

Because a search walks down a path that always heads towards the target, it can remember the best answer seen on the way. The **floor** of `x` (the largest value ≤ x) is the last node where the search went **right**, or `x` itself; the **ceiling** (smallest value ≥ x) is the last node where it went left.

Floors and ceilings answer questions like "which price band is this in?" and "what's the next appointment after 2 pm?".

**Range queries** list every value from `lo` to `hi`. An in-order walk that **skips** subtrees that can't contain any values in range does it in O(h + k) for k results, rather than visiting all n nodes: if a node's value is below `lo`, its whole left subtree is too, so don't go left; if it's above `hi`, don't go right.

## Shape depends on order

A BST's shape comes from the order its values were inserted (a simple insert walks down as if searching and attaches a new leaf where the search fell off; tomorrow covers insertion properly):

Inserting **sorted** data makes a chain of 1,022 edges: every search is O(n), and the tree is no better than a list. Random order gives 23 here: on average a random BST's height grows like about 3 log₂ n (Bruce Reed pinned the constant down in 2003), far less than n. Data often arrives sorted (dates, ids), which is exactly why Day 3's self-balancing trees exist. When you have all the values in advance, sort them and build the tree from the **middle** outwards: the middle value is the root, the middle of each half is its children, and so on, giving the smallest possible height.

## Practice

## Quiz

## Daily challenge

## Before you go

Insert the letters of your full name, one at a time, into an empty BST on paper (skipping repeats), then write its in-order traversal and check it's sorted. Tomorrow: inserting and deleting properly, including the tricky case of deleting a node with two children.